In [ ]:
# ── Mini blockchain ledger ────────────────────────────────────────────────

class Ledger:
    """
    Append-only hash-chained credential store.
    Each block: { index, timestamp, key_hash, encrypted, prev_hash, block_hash }
    The chain is valid if every block's prev_hash matches the previous block's block_hash.
    """

    def __init__(self, path: Path):
        self.path   = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.chain  = self._load()
        self._lock  = threading.Lock()

    def _load(self) -> list:
        if self.path.exists():
            try:
                chain = json.loads(self.path.read_text())
                if self._verify(chain):
                    log.info(f"Ledger loaded — {len(chain)} block(s), chain valid.")
                    return chain
                else:
                    log.warning("Ledger integrity check FAILED — starting fresh chain.")
            except (json.JSONDecodeError, KeyError):
                log.warning("Ledger corrupt — starting fresh chain.")
        log.info("No existing ledger — initialising genesis block.")
        return [self._genesis()]

    def _genesis(self) -> dict:
        return self._make_block(0, "GENESIS", b"", "0" * 64)

    def _make_block(self, index: int, key_hash: str, encrypted: bytes, prev_hash: str) -> dict:
        block = {
            "index":     index,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "key_hash":  key_hash,
            "encrypted": encrypted.hex() if isinstance(encrypted, bytes) else encrypted,
            "prev_hash": prev_hash,
        }
        block["block_hash"] = self._hash(block)
        return block

    @staticmethod
    def _hash(block: dict) -> str:
        payload = json.dumps({k: v for k, v in block.items() if k != "block_hash"}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()

    def _verify(self, chain: list) -> bool:
        for i in range(1, len(chain)):
            prev  = chain[i - 1]
            curr  = chain[i]
            if curr["prev_hash"] != prev["block_hash"]:
                return False
            expected = self._hash({k: v for k, v in curr.items() if k != "block_hash"})
            if curr["block_hash"] != expected:
                return False
        return True

    def append(self, key_hash: str, encrypted: bytes) -> dict:
        with self._lock:
            prev_hash = self.chain[-1]["block_hash"]
            block     = self._make_block(len(self.chain), key_hash, encrypted, prev_hash)
            self.chain.append(block)
            self.path.write_text(json.dumps(self.chain, indent=2))
            log.info(f"Block #{block['index']} committed — hash: {block['block_hash'][:16]}...")
            return block

    def is_valid(self) -> bool:
        return self._verify(self.chain)